# Complete Colab hyperparameter search — revised

This revision adds budget-capacity estimates for continuous execution, objective-aware sampler and study naming, explicit cost-rate handling, and a gated same-protocol reference benchmark. Expensive launch gates default to `False`.


## 1. Setup

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
REPO_URL="https://github.com/TrueRottweiler/WashingtonCsed504.git"; BRANCH="feature/hpo-framework"; REPO_ROOT=Path("/content/WashingtonCsed504")
if not (REPO_ROOT/"src/a1-cv/hpo").exists():
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO_URL,str(REPO_ROOT)],check=True)
CV_DIR=REPO_ROOT/"src/a1-cv"; os.chdir(CV_DIR); sys.path.insert(0,str(CV_DIR)) if str(CV_DIR) not in sys.path else None
subprocess.run([sys.executable,"-m","pip","install","-q","-r","hpo_requirements.txt"],check=True)
import torch, optuna, pandas as pd
print(sys.version); print(torch.__version__, torch.version.cuda); print(optuna.__version__)

## 2. Persistence and Google Drive

In [ ]:
MOUNT_DRIVE=True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT=Path("/content/drive/MyDrive/WashingtonCsed504-HPO")
else:
    PERSIST_ROOT=Path("/content/WashingtonCsed504-HPO")
PERSIST_ROOT.mkdir(parents=True,exist_ok=True)
RESUME=True
print("Persistence root:",PERSIST_ROOT)

## 3. Hardware profile, recommendations, and overrides

In [ ]:
from hpo.hardware import detect_hardware
from hpo.scheduler import plan_resources
hardware=detect_hardware(); print(json.dumps(hardware.to_dict(),indent=2,default=str))
DEVICE_OVERRIDE="auto"; CONCURRENT_TRIALS_OVERRIDE=None; INTRAOP_THREADS_OVERRIDE=None; INTEROP_THREADS_OVERRIDE=None; WORKERS_OVERRIDE=None; MEMORY_RESERVE_GB=1.0
resource_plan=plan_resources(hardware,device=DEVICE_OVERRIDE,requested_concurrency=CONCURRENT_TRIALS_OVERRIDE,requested_intraop_threads=INTRAOP_THREADS_OVERRIDE,requested_interop_threads=INTEROP_THREADS_OVERRIDE,requested_workers=WORKERS_OVERRIDE,memory_reserve_gb=MEMORY_RESERVE_GB)
print(json.dumps(resource_plan.to_dict(),indent=2))

## 4. Experiment selection

In [ ]:
DATASET="cifar10"  # cifar10, cifar100, imagenet32
MODEL="resnet18"   # resnet18, resnet50, vit, vit_base
if DATASET=="imagenet32":
    IMAGENET32_ROOT="/content/imagenet32"  # must already contain the repository-supported data
else: IMAGENET32_ROOT=None
NUM_CLASSES={"cifar10":10,"cifar100":100,"imagenet32":1000}[DATASET]
print(DATASET,MODEL,NUM_CLASSES)

## 5. Search-space source: built-in, Python dictionary, Python list, CSV upload, JSON/YAML, or manual input

In [ ]:
from pathlib import Path

from hpo.search_space import (
    normalize_space,
    load_csv,
    load_space,
    preview_rows,
    combination_count,
)
from hpo.notebook_api import (
    preview_dataframe,
    optional_widgets,
)


INPUT_SOURCE = "builtin"
# Options:
# builtin, dictionary, list, csv, csv_upload,
# json, yaml, manual


BUILTIN_CSV = CV_DIR / (
    "hpo_configs/search_spaces/vit_cifar.csv"
    if MODEL.startswith("vit")
    else
    "hpo_configs/search_spaces/resnet18_cifar.csv"
)


DICT_SPACE = {
    "learning_rate": {
        "type": "float",
        "low": 1e-4,
        "high": 0.2,
        "log": True,
        "default": 0.01,
    },
    "batch_size": {
        "type": "categorical",
        "choices": [64, 128, 256],
        "default": 128,
    },
    "optimizer": {
        "type": "categorical",
        "choices": ["sgd", "adamw"],
        "default": "sgd",
    },
    "momentum": {
        "type": "float",
        "low": 0.8,
        "high": 0.95,
        "step": 0.05,
        "default": 0.9,
        "condition": 'optimizer == "sgd"',
    },
    "beta1": {
        "type": "float",
        "low": 0.8,
        "high": 0.95,
        "step": 0.05,
        "default": 0.9,
        "condition": 'optimizer == "adamw"',
    },
}


LIST_SPACE = [
    {
        "name": name,
        **value,
    }
    for name, value in DICT_SPACE.items()
]


MANUAL_SPACE = DICT_SPACE.copy()


if INPUT_SOURCE in {"builtin", "csv"}:
    specs = load_csv(BUILTIN_CSV)

elif INPUT_SOURCE == "csv_upload":
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "CSV upload is available in Colab. "
            "Outside Colab, use INPUT_SOURCE='csv'."
        ) from exc

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(
            "Upload exactly one CSV search-space file."
        )

    from hpo.notebook_api import normalize_uploaded_csv

    filename, content = next(iter(uploaded.items()))

    specs = normalize_uploaded_csv(
        content,
        filename=filename,
    )

elif INPUT_SOURCE == "dictionary":
    specs = normalize_space(
        DICT_SPACE,
        source_name="dictionary cell",
    )

elif INPUT_SOURCE == "list":
    specs = normalize_space(
        LIST_SPACE,
        source_name="list cell",
    )

elif INPUT_SOURCE == "manual":
    specs = normalize_space(
        MANUAL_SPACE,
        source_name="manual cell",
    )

elif INPUT_SOURCE == "json":
    specs = load_space(
        Path("/content/search_space.json")
    )

elif INPUT_SOURCE == "yaml":
    specs = load_space(
        Path("/content/search_space.yaml")
    )

else:
    raise ValueError(
        f"Unsupported INPUT_SOURCE: {INPUT_SOURCE!r}"
    )


parameter_names = [
    spec.name
    for spec in specs
]

assert parameter_names, (
    "The normalized search space is empty."
)

assert "inline" not in parameter_names, (
    "Malformed inline search-space wrapper detected."
)


display(
    preview_dataframe(specs)
)

finite_combinations = combination_count(specs)

if finite_combinations is None:
    print(
        "Finite combinations: not finite or not "
        "meaningfully enumerable because the space "
        "contains continuous parameters."
    )
else:
    print(
        "Finite combinations:",
        finite_combinations,
    )

widgets = optional_widgets()

if widgets is None:
    display("ipywidgets unavailable")
else:
    display(widgets)

print(
    "Normalized parameters:",
    parameter_names,
)
print(
    "Search-space source:",
    INPUT_SOURCE,
)

In [ ]:

import hpo.search_space as search_space_module

print(
    "Loaded module:",
    search_space_module.__file__,
)

print(
    "load_csv exists:",
    hasattr(search_space_module, "load_csv"),
)

print(
    "load_space exists:",
    hasattr(search_space_module, "load_space"),
)

print(
    "load_json exists:",
    hasattr(search_space_module, "load_json"),
)

print(
    "load_yaml exists:",
    hasattr(search_space_module, "load_yaml"),
)

## 6. Validation and feasibility preview

In [ ]:
from hpo.constraints import validate_candidate
from hpo.exceptions import InvalidTrialError
warnings=[]
for spec in specs:
    if spec.name=="batch_size": warnings.append("Batch values will be filtered by calibration before launch.")
print("Conditions:",[s.condition for s in specs if s.condition])
print("Defaults:",{s.name:s.default for s in specs if s.default is not None})
print("Warnings:",warnings)
# Structural model constraints are checked before allocation: ViT divisibility/patch size, optimizer-specific fields, precision, and memory calibration.

## 7. Search mode and continuous execution

Choose one bounded search mode. Continuous execution wraps that mode in resumable sessions and must be limited by at least one stopping rule.


In [ ]:
MODE = "proxy"  # proxy, successive_halving, full
CONTINUOUS = False

CONTINUOUS_SETTINGS = {
    "enabled": CONTINUOUS,
    "strategy": MODE,
    "maximum_trials": 32 if CONTINUOUS else None,
    "maximum_wall_time_hours": 8.0 if CONTINUOUS else None,
    "maximum_gpu_hours": None,
    "maximum_cpu_hours": None,
    "maximum_cost_usd": None,
    "target_validation_metric": None,
    "stop_after_no_improvement_trials": 25,
    "minimum_improvement": 0.0005,
    "pareto_stagnation_trials": 25,
    "checkpoint_after_each_trial": True,
}

if CONTINUOUS and not any(
    CONTINUOUS_SETTINGS[name] is not None
    for name in (
        "maximum_trials",
        "maximum_wall_time_hours",
        "maximum_gpu_hours",
        "maximum_cpu_hours",
        "maximum_cost_usd",
        "target_validation_metric",
        "stop_after_no_improvement_trials",
        "pareto_stagnation_trials",
    )
):
    raise ValueError(
        "Continuous execution requires at least one stopping rule."
    )

print("Mode:", MODE)
print(json.dumps(CONTINUOUS_SETTINGS, indent=2))


## 8. Objectives, hard constraints, sampler identity, and cost rates

The objective profile determines the effective sampler and becomes part of the study name. Monetary cost remains unavailable until all required user rates are supplied.


In [ ]:
OBJECTIVE_PROFILE = "pareto_accuracy_time_memory"
# Options:
# - accuracy_only
# - pareto_accuracy_time
# - pareto_accuracy_time_memory

SINGLE_OBJECTIVE_SAMPLER = "tpe"  # tpe, random, qmc

if OBJECTIVE_PROFILE == "accuracy_only":
    OBJECTIVES = [
        {
            "name": "validation_top1",
            "direction": "maximize",
            "primary": True,
        },
    ]
    REQUESTED_SAMPLER = SINGLE_OBJECTIVE_SAMPLER
    OBJECTIVE_TAG = "accuracy"

elif OBJECTIVE_PROFILE == "pareto_accuracy_time":
    OBJECTIVES = [
        {
            "name": "validation_top1",
            "direction": "maximize",
            "primary": True,
        },
        {
            "name": "wall_seconds",
            "direction": "minimize",
            "primary": False,
        },
    ]
    REQUESTED_SAMPLER = "nsga2"
    OBJECTIVE_TAG = "pareto-accuracy-time"

elif OBJECTIVE_PROFILE == "pareto_accuracy_time_memory":
    OBJECTIVES = [
        {
            "name": "validation_top1",
            "direction": "maximize",
            "primary": True,
        },
        {
            "name": "wall_seconds",
            "direction": "minimize",
            "primary": False,
        },
        {
            "name": "peak_gpu_memory_mb",
            "direction": "minimize",
            "primary": False,
        },
    ]
    REQUESTED_SAMPLER = "nsga2"
    OBJECTIVE_TAG = "pareto-accuracy-time-memory"

else:
    raise ValueError(
        f"Unsupported OBJECTIVE_PROFILE: {OBJECTIVE_PROFILE!r}"
    )

EFFECTIVE_SAMPLER = (
    "nsga2" if len(OBJECTIVES) > 1 else REQUESTED_SAMPLER
)

CONSTRAINTS = [
    {
        "name": "peak_gpu_memory_mb",
        "operator": "<=",
        "value": 12000,
    }
]

COST_PROFILE = "unavailable"
# Options:
# - unavailable: record compute quantities but do not claim dollar cost
# - marginal_hourly: calculate marginal GPU, CPU, and storage cost

GPU_USD_PER_HOUR = None
CPU_USD_PER_HOUR = None
STORAGE_USD_PER_GB_MONTH = None
COLAB_SUBSCRIPTION_USD = None
COLAB_COMPUTE_UNIT_USD = None

if COST_PROFILE == "marginal_hourly":
    required_user_rates = {
        "GPU_USD_PER_HOUR": GPU_USD_PER_HOUR,
        "CPU_USD_PER_HOUR": CPU_USD_PER_HOUR,
        "STORAGE_USD_PER_GB_MONTH": STORAGE_USD_PER_GB_MONTH,
    }
    missing_user_rates = [
        name
        for name, value in required_user_rates.items()
        if value is None
    ]
    if missing_user_rates:
        raise ValueError(
            "Provide numeric marginal rates for: "
            + ", ".join(missing_user_rates)
        )
elif COST_PROFILE != "unavailable":
    raise ValueError(f"Unsupported COST_PROFILE: {COST_PROFILE!r}")

COST_RATES = {
    "gpu_usd_per_hour": (
        GPU_USD_PER_HOUR
        if COST_PROFILE == "marginal_hourly"
        else None
    ),
    "cpu_usd_per_hour": (
        CPU_USD_PER_HOUR
        if COST_PROFILE == "marginal_hourly"
        else None
    ),
    "storage_usd_per_gb_month": (
        STORAGE_USD_PER_GB_MONTH
        if COST_PROFILE == "marginal_hourly"
        else None
    ),
    "electricity_usd_per_kwh": None,
    "colab_subscription_usd": COLAB_SUBSCRIPTION_USD,
    "colab_compute_unit_usd": COLAB_COMPUTE_UNIT_USD,
}

REQUIRED_COST_FIELDS = (
    "gpu_usd_per_hour",
    "cpu_usd_per_hour",
    "storage_usd_per_gb_month",
)
MISSING_COST_RATES = [
    name
    for name in REQUIRED_COST_FIELDS
    if COST_RATES[name] is None
]
COST_ESTIMATION_AVAILABLE = not MISSING_COST_RATES

uses_cost_objective = any(
    objective["name"] == "estimated_cost_usd"
    for objective in OBJECTIVES
)
uses_cost_constraint = any(
    constraint["name"]
    in {"estimated_cost_usd", "known_component_total_usd"}
    for constraint in CONSTRAINTS
)

if (
    uses_cost_objective or uses_cost_constraint
) and not COST_ESTIMATION_AVAILABLE:
    raise ValueError(
        "Cost is configured as an objective or constraint, but "
        "required rates are missing: "
        + ", ".join(MISSING_COST_RATES)
    )

print(
    json.dumps(
        {
            "objective_profile": OBJECTIVE_PROFILE,
            "objectives": OBJECTIVES,
            "requested_sampler": REQUESTED_SAMPLER,
            "effective_sampler": EFFECTIVE_SAMPLER,
            "constraints": CONSTRAINTS,
            "cost_profile": COST_PROFILE,
            "cost_rates": COST_RATES,
            "cost_estimation_available": COST_ESTIMATION_AVAILABLE,
            "missing_cost_rates": MISSING_COST_RATES,
            "subscription_is_sunk": (
                COLAB_SUBSCRIPTION_USD is not None
            ),
            "compute_unit_note": (
                "The compute-unit rate is recorded but cannot be "
                "converted to cost unless compute-unit consumption "
                "is separately measured."
            ),
        },
        indent=2,
    )
)


## 9. Calibration: batch size, real training steps, validation, checkpoint, and memory

In [ ]:
from hpo.adapters import RepoModules, build_trial_model, build_trial_dataset
from hpo.calibration import calibrate_batch_sizes
modules=RepoModules(REPO_ROOT); device=torch.device(resource_plan.device)
dataset_cfg={"name":DATASET,"validation_fraction":0.1,"split_seed":42}; dataset_cfg.update({"root":IMAGENET32_ROOT} if IMAGENET32_ROOT else {})
bundle=build_trial_dataset(modules,dataset_cfg,{"seed":42},device)
model_cfg={"name":MODEL}
def make_batch(bs):
    x,y=next(bundle.train.epoch(bs,train=True)); return x,y
calibration=calibrate_batch_sizes(lambda: build_trial_model(modules,model_cfg,bundle.num_classes,device),make_batch,device=device,candidates=[32,64,128,256,512],precision="bf16" if hardware.bf16_native else "fp16" if device.type=="cuda" else "fp32",warmup_steps=2,measure_steps=3,channels_last=MODEL.startswith("resnet"))
print(json.dumps(calibration.to_dict(),indent=2))

Actually filter batch sizes using calibration

In [ ]:
from dataclasses import replace


batch_spec = next(
    spec
    for spec in specs
    if spec.name == "batch_size"
)

completed_batches = {
    measurement.batch_size
    for measurement in calibration.measurements
    if measurement.status == "completed"
}

recommended_batches = [
    batch_size
    for batch_size in batch_spec.choices
    if (
        batch_size
        in calibration.recommended_candidates
        and batch_size in completed_batches
    )
]

fitting_search_batches = [
    batch_size
    for batch_size in batch_spec.choices
    if batch_size in completed_batches
]

safe_batch_choices = (
    recommended_batches
    or fitting_search_batches
)

if not safe_batch_choices:
    raise RuntimeError(
        "None of the configured search batch sizes "
        "completed calibration."
    )

preferred_default = (
    calibration.highest_throughput_batch
    if calibration.highest_throughput_batch
    in safe_batch_choices
    else safe_batch_choices[-1]
)

specs = [
    replace(
        spec,
        choices=tuple(safe_batch_choices),
        default=preferred_default,
    )
    if spec.name == "batch_size"
    else spec
    for spec in specs
]

print(
    "Original batch choices:",
    batch_spec.choices,
)
print(
    "Completed calibration batches:",
    sorted(completed_batches),
)
print(
    "Recommended search batches:",
    safe_batch_choices,
)
print(
    "New batch default:",
    preferred_default,
)

## 10. Build the resolved configuration and pre-search estimate

Bounded searches receive a fixed workload estimate. Continuous searches receive a budget-capacity projection describing how many candidates or sessions may fit before an enabled stopping rule is reached.


In [ ]:
import copy
import yaml
from dataclasses import asdict

from hpo.config import load_study_config

try:
    from hpo.estimation import (
        estimate_search,
        estimate_continuous_capacity,
    )
except ImportError as exc:
    raise RuntimeError(
        "The cloned branch does not contain "
        "estimate_continuous_capacity(). Push the updated "
        "src/a1-cv/hpo/estimation.py to the same branch used "
        "by this notebook."
    ) from exc

SEARCH_SEED = 42
SPLIT_SEED = 42
RUN_NUMBER = 2
EXECUTION_TAG = "continuous" if CONTINUOUS else "bounded"
RUN_TAG = (
    f"{OBJECTIVE_TAG}-{EFFECTIVE_SAMPLER}-"
    f"s{SEARCH_SEED}-r{RUN_NUMBER:02d}"
)

STUDY_NAME = (
    f"{MODEL}-{DATASET}-{MODE}-{EXECUTION_TAG}-{RUN_TAG}"
)
CONFIG_PATH = PERSIST_ROOT / f"{STUDY_NAME}.yaml"

base = yaml.safe_load(
    (
        CV_DIR
        / (
            "hpo_configs/colab/vit_cifar10.yaml"
            if MODEL.startswith("vit")
            else "hpo_configs/colab/"
            "resnet18_cifar10_successive_halving.yaml"
        )
    ).read_text(encoding="utf-8")
)

base["study"].update(
    {
        "name": STUDY_NAME,
        "output_dir": str(PERSIST_ROOT),
        "storage_path": str(
            PERSIST_ROOT / f"{STUDY_NAME}.db"
        ),
        "resume": RESUME,
        "seed": SEARCH_SEED,
    }
)
base.setdefault("search", {})["mode"] = MODE
base["search"]["sampler"] = EFFECTIVE_SAMPLER

base["dataset"].update(dataset_cfg)
base["dataset"]["split_seed"] = SPLIT_SEED
base["model"] = {"name": MODEL}
base["objectives"] = OBJECTIVES
base["constraints"] = CONSTRAINTS
base["cost_rates"] = COST_RATES
base["continuous"] = CONTINUOUS_SETTINGS

SELECTED_PRECISION = (
    "bf16"
    if hardware.bf16_native
    else "fp16"
    if hardware.cuda_available
    else "fp32"
)

base["runtime"].update(
    {
        "device": resource_plan.device,
        "precision": SELECTED_PRECISION,
        "tf32": bool(hardware.tf32_supported),
        "channels_last": MODEL.startswith("resnet"),
        "concurrent_trials": resource_plan.concurrent_trials,
        "intraop_threads": resource_plan.intraop_threads,
        "interop_threads": resource_plan.interop_threads,
        "workers": resource_plan.workers_per_trial,
        "memory_reserve_gb": max(MEMORY_RESERVE_GB, 1.5),
    }
)

base["search_space"] = {
    spec.name: {
        key: value
        for key, value in spec.to_dict().items()
        if key not in {"name", "source", "item"}
        and value not in (None, [], ())
    }
    for spec in specs
}

CONFIG_PATH.write_text(
    yaml.safe_dump(base, sort_keys=False),
    encoding="utf-8",
)
config = load_study_config(CONFIG_PATH)

assert config.mode == MODE, (
    f"Expected mode {MODE!r}, got {config.mode!r}"
)
assert config.sampler == EFFECTIVE_SAMPLER
assert config.continuous.enabled is CONTINUOUS
assert config.dataset.get("split_seed") == SPLIT_SEED
assert config.seed == SEARCH_SEED
assert OBJECTIVE_TAG in STUDY_NAME
assert EFFECTIVE_SAMPLER in STUDY_NAME
assert all(
    spec.name != "inline"
    for spec in config.search_space
)
if len(config.objectives) > 1:
    assert EFFECTIVE_SAMPLER == "nsga2"

if calibration.highest_throughput_batch is None:
    failures = [
        measurement.__dict__
        for measurement in calibration.measurements
    ]
    raise RuntimeError(
        "No calibration batch completed successfully. "
        f"Measurements: {failures}"
    )

measurement = next(
    item
    for item in calibration.measurements
    if item.batch_size
    == calibration.highest_throughput_batch
)

calibration_record = {
    "batch_size": measurement.batch_size,
    "seconds_per_example": (
        measurement.seconds_per_step
        / measurement.batch_size
    ),
    "evaluation_seconds_per_epoch": 0.0,
    "checkpoint_seconds_per_epoch": 0.0,
    "checkpoint_size_mb": 0.0,
    "peak_memory_mb": (
        measurement.peak_allocated_mb or 0.0
    ),
    "cpu_seconds": 0.0,
    "elapsed_seconds": measurement.seconds_per_step,
}

estimate_arguments = {
    "train_examples": bundle.train_examples,
    "validation_examples": bundle.validation_examples,
    "calibration_records": [calibration_record],
    "representative_batch_size": measurement.batch_size,
}

if config.continuous.enabled:
    estimate_payload = estimate_continuous_capacity(
        config,
        **estimate_arguments,
    )
else:
    estimate_payload = estimate_search(
        config,
        **estimate_arguments,
    ).to_dict()

ESTIMATE_PATH = (
    PERSIST_ROOT
    / f"{STUDY_NAME}-pre_search_estimate.json"
)
ESTIMATE_PATH.write_text(
    json.dumps(estimate_payload, indent=2),
    encoding="utf-8",
)

resolved_summary = {
    "study_name": config.name,
    "mode": config.mode,
    "continuous": config.continuous.enabled,
    "search_seed": config.seed,
    "split_seed": config.dataset.get("split_seed"),
    "effective_sampler": config.sampler,
    "objective_names": [
        objective.name for objective in config.objectives
    ],
    "cost_estimation_available": COST_ESTIMATION_AVAILABLE,
    "device": config.runtime.device,
    "precision": config.runtime.precision,
    "tf32": config.runtime.tf32,
    "channels_last": config.runtime.channels_last,
    "concurrent_trials": config.runtime.concurrent_trials,
    "batch_choices": next(
        list(spec.choices)
        for spec in config.search_space
        if spec.name == "batch_size"
    ),
    "proxy_trials": config.proxy.trials,
    "proxy_budget": asdict(config.proxy.budget),
    "rung_budgets": config.successive_halving.rung_budgets,
    "full_trials": config.full.trials,
    "full_budget": asdict(config.full.budget),
    "estimate_classification": estimate_payload.get(
        "classification"
    ),
}

print("Resolved configuration:")
print(json.dumps(resolved_summary, indent=2, default=str))
print("Pre-search estimate:")
print(json.dumps(estimate_payload, indent=2, default=str))
print("Configuration path:", CONFIG_PATH)
print("Estimate path:", ESTIMATE_PATH)


## 11. Explicit launch gate and search execution with live monitoring

Review the resolved configuration and estimate before changing `START_SEARCH` to `True`.


In [ ]:
START_SEARCH = False

if START_SEARCH:
    from hpo.monitoring import run_command_with_monitor

    study_dir = PERSIST_ROOT / STUDY_NAME
    log_path = PERSIST_ROOT / f"{STUDY_NAME}.log"
    command = [
        sys.executable,
        "-m",
        "hpo.cli",
        "--repo-root",
        str(REPO_ROOT),
        "search",
        "--config",
        str(CONFIG_PATH),
        "--mode",
        MODE,
    ]
    if CONTINUOUS:
        command.append("--continuous")

    expected_records = None if CONTINUOUS else (
        config.full.trials
        * (len(config.full.budget.seeds) + 1)
        if MODE == "full"
        else config.proxy.trials
    )

    return_code = run_command_with_monitor(
        command,
        cwd=CV_DIR,
        study_dir=study_dir,
        log_path=log_path,
        interval_seconds=15,
        timeout_seconds=None,
        expected_records=expected_records,
    )
    print("Return code:", return_code)
    if return_code != 0:
        print("Log tail:")
        print(
            "\n".join(
                log_path.read_text(
                    encoding="utf-8"
                ).splitlines()[-120:]
            )
        )
        raise RuntimeError(
            "Search failed; inspect the log above."
        )
else:
    print(
        "Search not started. Set START_SEARCH=True only "
        "after reviewing the resolved configuration and estimate."
    )


## 12. Resume after reconnection

This cell actively continues an unfinished study. Leave `RESUME_SEARCH=False` during ordinary notebook review or `Run all`.


In [ ]:
# Reconnect, remount Drive, and rerun setup/configuration cells first.
RESUME_SEARCH = False

if RESUME_SEARCH:
    from hpo.study import HpoStudy

    resumed = HpoStudy(
        CONFIG_PATH,
        repo_root=REPO_ROOT,
    ).run()
    print(json.dumps(resumed, indent=2, default=str))
else:
    print(
        "Resume disabled. Completed trials remain stored in "
        "SQLite, JSONL, state JSON, and checkpoints."
    )


## 13. Results analysis, Pareto selection, and explicit cost availability

Cost-based selection is skipped when the required marginal rates are unavailable. Compute quantities remain reportable.


In [ ]:
from hpo.costing import estimate_cost
from hpo.persistence import read_jsonl
from hpo.reporting import export_reports
from hpo.schemas import ObjectiveSpec
from hpo.selection import (
    fastest_above_accuracy,
    lowest_cost_within_accuracy_margin,
    lowest_memory_above_accuracy,
    pareto_knee,
)

study_dir = PERSIST_ROOT / STUDY_NAME

if (study_dir / "trials.jsonl").exists():
    rows = read_jsonl(study_dir / "trials.jsonl")
    objectives = [
        ObjectiveSpec(**objective)
        for objective in OBJECTIVES
    ]
    report = export_reports(study_dir, objectives)
    completed = [
        row
        for row in rows
        if row.get("status") == "completed"
        and row.get("metrics")
    ]

    print(json.dumps(report, indent=2, default=str))
    display(pd.read_csv(study_dir / "all_trials.csv"))
    display(pd.read_csv(study_dir / "pareto_trials.csv"))

    print(
        "Fastest acceptable:",
        fastest_above_accuracy(
            completed,
            minimum_accuracy=0.80,
        ),
    )
    print(
        "Lowest memory acceptable:",
        lowest_memory_above_accuracy(
            completed,
            minimum_accuracy=0.80,
        ),
    )

    if COST_ESTIMATION_AVAILABLE:
        cost_adjusted_rows = copy.deepcopy(completed)
        for row in cost_adjusted_rows:
            row["metrics"].update(
                estimate_cost(
                    row["metrics"],
                    config.cost_rates,
                )
            )
        print(
            "Lowest cost near best:",
            lowest_cost_within_accuracy_margin(
                cost_adjusted_rows,
                accuracy_margin=0.01,
            ),
        )
    else:
        print(
            "Lowest-cost selection skipped. Missing rates:",
            MISSING_COST_RATES,
        )
        print(
            "GPU-hours, CPU-hours, storage, and known "
            "component costs remain available."
        )

    print(
        "Pareto knee:",
        pareto_knee(completed, objectives),
    )
else:
    print("Run or resume a study first.")


## 14. Historical context and same-protocol reference benchmark

Historical repository results are retained as context but are labeled not directly comparable. The optional reference track reruns the repository recipe through the HPO adapter with the same validation split, seeds, full budget, hardware settings, and no test-set evaluation.


In [ ]:
from hpo.baselines import (
    REPOSITORY_RECIPES,
    load_repository_baselines,
)
from hpo.benchmark import normalized_parameter_distance
from hpo.persistence import read_jsonl
from hpo.study import HpoStudy

RUN_REFERENCE_TRACK = False
REFERENCE_CONFIG_PATH = None
FAIR_COMPARISON_PATH = None

historical_baselines = [
    baseline
    for baseline in load_repository_baselines(REPO_ROOT)
    if baseline.dataset == DATASET
    and baseline.model == MODEL
]
historical_reference = (
    max(
        historical_baselines,
        key=lambda baseline: (
            baseline.validation_top1
            if baseline.validation_top1 is not None
            else float("-inf")
        ),
    )
    if historical_baselines
    else None
)

if historical_reference is not None:
    print(
        "Historical repository context (not directly comparable):"
    )
    print(
        json.dumps(
            {
                "name": historical_reference.name,
                "recorded_score": historical_reference.validation_top1,
                "seconds": historical_reference.seconds,
                "source": historical_reference.source,
                "directly_comparable": False,
                "reason": (
                    "The historical run used the final test split "
                    "during training-time evaluation and may differ "
                    "in hardware, seeds, and execution protocol."
                ),
            },
            indent=2,
            default=str,
        )
    )
else:
    print("No historical repository baseline was found.")

recipe = REPOSITORY_RECIPES.get((DATASET, MODEL))

if recipe is None:
    print(
        "No registered same-protocol reference recipe exists for "
        f"{MODEL} on {DATASET}. Historical context only."
    )
else:
    configured_parameter_names = {
        spec.name for spec in config.search_space
    }

    reference_params = {
        name: value
        for name, value in recipe.items()
        if name in configured_parameter_names
        and name != "epochs"
        and name != "augmentation"
    }

    if (
        "strong_augmentation"
        in configured_parameter_names
        and "augmentation" in recipe
    ):
        reference_params["strong_augmentation"] = (
            recipe["augmentation"] == "strong"
        )

    if "channels_last" in configured_parameter_names:
        reference_params["channels_last"] = (
            MODEL.startswith("resnet")
        )

    reference_batch_size = reference_params.get("batch_size")
    if (
        reference_batch_size is not None
        and reference_batch_size not in completed_batches
    ):
        raise RuntimeError(
            "The exact reference batch size did not complete "
            "calibration on this hardware. Do not silently "
            "substitute another batch size for the benchmark."
        )

    REFERENCE_STUDY_NAME = (
        f"{MODEL}-{DATASET}-full-reference-"
        f"split{SPLIT_SEED}-s{SEARCH_SEED}"
    )
    REFERENCE_CONFIG_PATH = (
        PERSIST_ROOT / f"{REFERENCE_STUDY_NAME}.yaml"
    )

    reference_base = copy.deepcopy(base)
    reference_base["study"].update(
        {
            "name": REFERENCE_STUDY_NAME,
            "output_dir": str(PERSIST_ROOT),
            "storage_path": str(
                PERSIST_ROOT / f"{REFERENCE_STUDY_NAME}.db"
            ),
            "seed": SEARCH_SEED,
            "resume": True,
        }
    )
    reference_base.setdefault("search", {})["mode"] = "full"
    reference_base["search"]["sampler"] = "random"
    reference_base["continuous"]["enabled"] = False
    reference_base["dataset"]["split_seed"] = SPLIT_SEED
    reference_base["objectives"] = [
        {
            "name": "validation_top1",
            "direction": "maximize",
            "primary": True,
        }
    ]
    reference_base["constraints"] = []
    reference_base["search_space"] = {
        name: {
            "type": "fixed",
            "default": value,
        }
        for name, value in reference_params.items()
    }
    reference_base["proxy"]["enabled"] = False
    reference_base["successive_halving"]["enabled"] = False
    reference_base["full"] = {
        "enabled": True,
        "trials": 1,
        "budget": {
            "epochs": config.full.budget.epochs,
            "max_steps": config.full.budget.max_steps,
            "data_fraction": config.full.budget.data_fraction,
            "validation_fraction": (
                config.full.budget.validation_fraction
            ),
            "validation_interval": (
                config.full.budget.validation_interval
            ),
            "seeds": list(config.full.budget.seeds),
        },
        "exhaustive": False,
        "maximum_combinations": 1,
        "allow_performance_pruning": False,
        "allow_safety_termination": True,
        "evaluate_test": False,
    }

    REFERENCE_CONFIG_PATH.write_text(
        yaml.safe_dump(reference_base, sort_keys=False),
        encoding="utf-8",
    )
    reference_config = load_study_config(
        REFERENCE_CONFIG_PATH
    )

    assert reference_config.mode == "full"
    assert not reference_config.continuous.enabled
    assert not reference_config.full.evaluate_test
    assert (
        reference_config.dataset.get("split_seed")
        == config.dataset.get("split_seed")
    )
    assert (
        reference_config.full.budget.seeds
        == config.full.budget.seeds
    )
    assert (
        reference_config.full.budget.epochs
        == config.full.budget.epochs
    )

    print("Same-protocol reference configuration:")
    print(
        json.dumps(
            {
                "study_name": REFERENCE_STUDY_NAME,
                "config_path": str(REFERENCE_CONFIG_PATH),
                "params": reference_params,
                "epochs": reference_config.full.budget.epochs,
                "seeds": reference_config.full.budget.seeds,
                "split_seed": reference_config.dataset.get(
                    "split_seed"
                ),
                "precision": reference_config.runtime.precision,
                "evaluate_test": (
                    reference_config.full.evaluate_test
                ),
            },
            indent=2,
            default=str,
        )
    )

    if RUN_REFERENCE_TRACK:
        reference_summary = HpoStudy(
            REFERENCE_CONFIG_PATH,
            repo_root=REPO_ROOT,
        ).run()
        print(
            json.dumps(
                reference_summary,
                indent=2,
                default=str,
            )
        )
    else:
        print(
            "Reference track not started. Set "
            "RUN_REFERENCE_TRACK=True only after reviewing "
            "the fixed configuration above."
        )

    discovery_results_path = (
        PERSIST_ROOT / STUDY_NAME / "trials.jsonl"
    )
    reference_results_path = (
        PERSIST_ROOT
        / REFERENCE_STUDY_NAME
        / "trials.jsonl"
    )

    discovery_rows = (
        read_jsonl(discovery_results_path)
        if discovery_results_path.exists()
        else []
    )
    reference_rows = (
        read_jsonl(reference_results_path)
        if reference_results_path.exists()
        else []
    )

    discovery_full_rows = [
        row
        for row in discovery_rows
        if row.get("status") == "completed"
        and row.get("stage") == "full"
        and row.get("metrics")
    ]
    reference_full_rows = [
        row
        for row in reference_rows
        if row.get("status") == "completed"
        and row.get("stage") == "full"
        and row.get("metrics")
    ]

    if discovery_full_rows and reference_full_rows:
        best_discovered = max(
            discovery_full_rows,
            key=lambda row: row["metrics"].get(
                "validation_top1",
                float("-inf"),
            ),
        )
        same_protocol_reference = max(
            reference_full_rows,
            key=lambda row: row["metrics"].get(
                "validation_top1",
                float("-inf"),
            ),
        )

        discovered_metrics = best_discovered["metrics"]
        reference_metrics = same_protocol_reference["metrics"]

        fair_comparison = {
            "comparison_type": (
                "same-protocol validation comparison"
            ),
            "directly_comparable": True,
            "protocol": {
                "dataset": DATASET,
                "model": MODEL,
                "validation_fraction": config.dataset.get(
                    "validation_fraction"
                ),
                "split_seed": SPLIT_SEED,
                "epochs": config.full.budget.epochs,
                "seeds": list(config.full.budget.seeds),
                "device": config.runtime.device,
                "precision": config.runtime.precision,
                "channels_last": config.runtime.channels_last,
                "evaluate_test": False,
            },
            "discovered": {
                "candidate_id": best_discovered.get(
                    "candidate_id"
                ),
                "validation_top1_mean": discovered_metrics.get(
                    "validation_top1"
                ),
                "validation_top1_min": discovered_metrics.get(
                    "validation_top1_min"
                ),
                "validation_top1_max": discovered_metrics.get(
                    "validation_top1_max"
                ),
                "wall_seconds": discovered_metrics.get(
                    "wall_seconds"
                ),
                "gpu_hours": discovered_metrics.get(
                    "gpu_hours"
                ),
                "peak_gpu_memory_mb": discovered_metrics.get(
                    "peak_gpu_memory_mb"
                ),
                "params": best_discovered.get("params"),
            },
            "reference": {
                "candidate_id": same_protocol_reference.get(
                    "candidate_id"
                ),
                "validation_top1_mean": reference_metrics.get(
                    "validation_top1"
                ),
                "validation_top1_min": reference_metrics.get(
                    "validation_top1_min"
                ),
                "validation_top1_max": reference_metrics.get(
                    "validation_top1_max"
                ),
                "wall_seconds": reference_metrics.get(
                    "wall_seconds"
                ),
                "gpu_hours": reference_metrics.get(
                    "gpu_hours"
                ),
                "peak_gpu_memory_mb": reference_metrics.get(
                    "peak_gpu_memory_mb"
                ),
                "params": reference_params,
            },
            "validation_accuracy_gap": (
                discovered_metrics.get("validation_top1")
                - reference_metrics.get("validation_top1")
            ),
            "parameter_distance": normalized_parameter_distance(
                best_discovered.get("params", {}),
                reference_params,
            ),
        }

        if historical_reference is not None:
            fair_comparison[
                "historical_repository_context"
            ] = {
                "directly_comparable": False,
                "name": historical_reference.name,
                "recorded_score": (
                    historical_reference.validation_top1
                ),
                "source": historical_reference.source,
                "reason": (
                    "Historical training evaluated against the "
                    "final test split and may differ in hardware, "
                    "seeds, and execution protocol."
                ),
            }

        FAIR_COMPARISON_PATH = (
            PERSIST_ROOT
            / f"{STUDY_NAME}-fair-reference-comparison.json"
        )
        FAIR_COMPARISON_PATH.write_text(
            json.dumps(
                fair_comparison,
                indent=2,
                default=str,
            ),
            encoding="utf-8",
        )

        print("Fair reference comparison:")
        print(
            json.dumps(
                fair_comparison,
                indent=2,
                default=str,
            )
        )
    else:
        print(
            "A fair comparison requires at least one completed "
            "full discovery result and one completed full "
            "reference result."
        )


## 15. Export selected hyperparameters and study artifacts

The export includes the resolved study, estimate, and—when available—the same-protocol reference configuration and comparison.


In [ ]:
EXPORT = True

if EXPORT and (PERSIST_ROOT / STUDY_NAME).exists():
    export_dir = PERSIST_ROOT / f"{STUDY_NAME}-export"
    export_dir.mkdir(parents=True, exist_ok=True)

    for name in [
        "all_trials.csv",
        "pareto_trials.csv",
        "best_validation_configuration.json",
        "pareto_knee_configuration.json",
        "report_summary.json",
        "environment.json",
        "resolved_config.json",
        "parameter_importance.json",
        "optimization_history.json",
        "optuna_trials.csv",
    ]:
        source = PERSIST_ROOT / STUDY_NAME / name
        if source.exists():
            shutil.copy2(source, export_dir / name)

    shutil.copy2(
        CONFIG_PATH,
        export_dir / CONFIG_PATH.name,
    )
    shutil.copy2(
        ESTIMATE_PATH,
        export_dir / ESTIMATE_PATH.name,
    )

    if (
        REFERENCE_CONFIG_PATH is not None
        and REFERENCE_CONFIG_PATH.exists()
    ):
        shutil.copy2(
            REFERENCE_CONFIG_PATH,
            export_dir / REFERENCE_CONFIG_PATH.name,
        )

    if (
        FAIR_COMPARISON_PATH is not None
        and FAIR_COMPARISON_PATH.exists()
    ):
        shutil.copy2(
            FAIR_COMPARISON_PATH,
            export_dir / FAIR_COMPARISON_PATH.name,
        )

    notebook_metadata = {
        "study_name": STUDY_NAME,
        "mode": MODE,
        "continuous": CONTINUOUS,
        "objective_profile": OBJECTIVE_PROFILE,
        "effective_sampler": EFFECTIVE_SAMPLER,
        "search_seed": SEARCH_SEED,
        "split_seed": SPLIT_SEED,
        "cost_profile": COST_PROFILE,
        "cost_estimation_available": COST_ESTIMATION_AVAILABLE,
        "missing_cost_rates": MISSING_COST_RATES,
        "historical_baseline_directly_comparable": False,
        "same_protocol_reference_available": (
            FAIR_COMPARISON_PATH is not None
            and FAIR_COMPARISON_PATH.exists()
        ),
    }
    (
        export_dir / "notebook_experiment_metadata.json"
    ).write_text(
        json.dumps(notebook_metadata, indent=2),
        encoding="utf-8",
    )

    archive = shutil.make_archive(
        str(export_dir),
        "zip",
        root_dir=export_dir,
    )
    print("Export:", archive)
else:
    print("Nothing to export yet.")


## Notes and limitations

- Search estimates are projections, not guarantees.
- Continuous-mode estimates describe budget capacity, not a fixed completion workload; stopping checks occur between bounded sessions and may overshoot by one session.
- Colab GPU assignment and session duration vary at runtime.
- One active trial per GPU is the default; multi-trial one-GPU execution requires measured benefit.
- Multiple objectives use NSGA-II; single-objective runs use the explicitly selected sampler.
- Monetary cost is unavailable until the required user-supplied GPU, CPU, and storage rates are present. Subscription cost is treated as sunk, and compute-unit cost is not inferred without measured compute-unit consumption.
- Historical repository results are context only because their evaluation protocol differs. Direct comparison requires the gated same-protocol reference track.
- Multi-objective staged promotion uses the designated primary metric; Pareto selection is applied to completed survivors.
- “Best” means best observed for the declared split, search space, fidelity, objectives, constraints, seeds, and budget.
- The final CIFAR test split remains reserved until model selection and final confirmation.
